# Notebook 01 — Document Pipeline Demo

## Project 01 — RAG Document Intelligence

This notebook demonstrates the first end-to-end document pipeline for the Brazil Agro-Environmental AI Portfolio.

The project uses only two source families:

1. **MapBiomas**
2. **INPE / TerraBrasilis**

The pipeline is:

```text
PDF documents
→ page-level text extraction
→ text cleaning
→ document chunking
→ document_chunks.parquet
→ ChromaDB vector store
→ retrieval test
→ golden questions
→ retrieval evaluation
```

This notebook is intentionally simple and robust. If no PDF files are found, it creates a small synthetic dataset so the pipeline can be tested before real documents are added.


## 1. Setup

Run this notebook from either:

```text
01-rag-document-intelligence/
```

or:

```text
01-rag-document-intelligence/notebooks/
```

The cell below makes `src/` importable.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()

# If running from notebooks/, move back to the repository root
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)


Project root: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence
Source directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\src


## 2. Check project directories

This uses the path utilities created in the project package.


In [2]:
from agro_rag.utils.paths import (
    ensure_project_directories,
    MAPBIOMAS_RAW_DIR,
    INPE_RAW_DIR,
    PARSED_DOCUMENTS_DIR,
    PROCESSED_DATA_DIR,
    CHROMA_DB_DIR,
)

ensure_project_directories()

print("Project directories are ready.")
print("MapBiomas raw directory:", MAPBIOMAS_RAW_DIR)
print("INPE raw directory:", INPE_RAW_DIR)
print("Parsed documents directory:", PARSED_DOCUMENTS_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("ChromaDB directory:", CHROMA_DB_DIR)


Project directories are ready.
MapBiomas raw directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\data\raw\mapbiomas
INPE raw directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\data\raw\inpe
Parsed documents directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\data\interim\parsed_documents
Processed data directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\data\processed
ChromaDB directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\chroma_db


## 3. Prepare document folders

Place PDF files in:

```text
data/raw/mapbiomas/documents/
data/raw/inpe/documents/
```

Examples:

- MapBiomas annual reports;
- MapBiomas methodology documents;
- INPE / TerraBrasilis / PRODES methodology documents;
- INPE deforestation monitoring reports.

The notebook searches recursively inside these folders.


In [3]:
MAPBIOMAS_DOCS_DIR = MAPBIOMAS_RAW_DIR / "documents"
INPE_DOCS_DIR = INPE_RAW_DIR / "documents"

MAPBIOMAS_DOCS_DIR.mkdir(parents=True, exist_ok=True)
INPE_DOCS_DIR.mkdir(parents=True, exist_ok=True)

mapbiomas_pdfs = sorted(MAPBIOMAS_DOCS_DIR.glob("**/*.pdf"))
inpe_pdfs = sorted(INPE_DOCS_DIR.glob("**/*.pdf"))

print(f"MapBiomas PDF files found: {len(mapbiomas_pdfs)}")
print(f"INPE PDF files found: {len(inpe_pdfs)}")

for pdf_path in mapbiomas_pdfs[:5]:
    print("MapBiomas:", pdf_path)

for pdf_path in inpe_pdfs[:5]:
    print("INPE:", pdf_path)


MapBiomas PDF files found: 0
INPE PDF files found: 0


## 4. Parse PDF files

This step extracts text page by page.

Output files:

```text
data/interim/parsed_documents/mapbiomas_pages.parquet
data/interim/parsed_documents/inpe_pages.parquet
```


In [4]:
from agro_rag.processing.parse_pdf import parse_and_save_pdf_folder

mapbiomas_pages_path = PARSED_DOCUMENTS_DIR / "mapbiomas_pages.parquet"
inpe_pages_path = PARSED_DOCUMENTS_DIR / "inpe_pages.parquet"

if mapbiomas_pdfs:
    mapbiomas_pages = parse_and_save_pdf_folder(
        folder_path=MAPBIOMAS_DOCS_DIR,
        output_path=mapbiomas_pages_path,
        source="MapBiomas",
        theme="land use and land cover",
        language="pt-BR",
        recursive=True,
    )
else:
    mapbiomas_pages = pd.DataFrame()

if inpe_pdfs:
    inpe_pages = parse_and_save_pdf_folder(
        folder_path=INPE_DOCS_DIR,
        output_path=inpe_pages_path,
        source="INPE",
        theme="deforestation monitoring",
        language="pt-BR",
        recursive=True,
    )
else:
    inpe_pages = pd.DataFrame()

print("MapBiomas parsed pages:", len(mapbiomas_pages))
print("INPE parsed pages:", len(inpe_pages))


MapBiomas parsed pages: 0
INPE parsed pages: 0


## 5. Synthetic fallback

Use this only when there are no real PDFs yet.

The synthetic dataset lets us test the whole pipeline without depending on external documents.


In [5]:
def build_synthetic_pages() -> pd.DataFrame:
    rows = [
        {
            "source": "MapBiomas",
            "document_name": "synthetic_mapbiomas_land_use_note",
            "file_path": "synthetic",
            "page_number": 1,
            "year": 2024,
            "theme": "land use and land cover",
            "language": "en",
            "text": (
                "MapBiomas provides annual land use and land cover information. "
                "It can support analysis of forest area, native vegetation, agriculture, "
                "pasture, and land-use transitions. These indicators support environmental "
                "screening, but they do not prove illegal deforestation or legal non-compliance."
            ),
        },
        {
            "source": "INPE",
            "document_name": "synthetic_inpe_prodes_note",
            "file_path": "synthetic",
            "page_number": 1,
            "year": 2024,
            "theme": "deforestation monitoring",
            "language": "en",
            "text": (
                "INPE and TerraBrasilis provide official deforestation monitoring evidence. "
                "PRODES is commonly used for annual deforestation monitoring. "
                "DETER is alert-oriented and should not be interpreted in the same way as PRODES. "
                "Deforestation evidence should be interpreted with methodological documentation."
            ),
        },
        {
            "source": "Project limitations",
            "document_name": "synthetic_limitations_note",
            "file_path": "synthetic",
            "page_number": 1,
            "year": 2024,
            "theme": "limitations",
            "language": "en",
            "text": (
                "This RAG system is designed for evidence organization and environmental screening. "
                "It does not determine illegal deforestation, legal responsibility, rural property "
                "compliance, or final ESG certification. A high risk score means deeper review is "
                "recommended, not that environmental crime was proven."
            ),
        },
    ]

    df = pd.DataFrame(rows)
    df["text_length"] = df["text"].str.len()
    return df


if mapbiomas_pages.empty and inpe_pages.empty:
    pages_df = build_synthetic_pages()
    synthetic_pages_path = PARSED_DOCUMENTS_DIR / "synthetic_pages.parquet"
    pages_df.to_parquet(synthetic_pages_path, index=False)
    print("No PDFs found. Synthetic demo pages were created.")
else:
    pages_df = pd.concat(
        [df for df in [mapbiomas_pages, inpe_pages] if not df.empty],
        ignore_index=True,
    )

print("Total page records:", len(pages_df))
pages_df.head()


No PDFs found. Synthetic demo pages were created.
Total page records: 3


,source,document_name,file_path,page_number,year,theme,language,text,text_length
0,MapBiomas,synthetic_mapbiomas_land_use_note,synthetic,1,2024,land use and land cover,en,MapBiomas provides annual land use and land co...,288
1,INPE,synthetic_inpe_prodes_note,synthetic,1,2024,deforestation monitoring,en,INPE and TerraBrasilis provide official defore...,296
2,Project limitations,synthetic_limitations_note,synthetic,1,2024,limitations,en,This RAG system is designed for evidence organ...,299


## 6. Clean extracted text

This step creates a `clean_text` column.

Cleaning includes:

- whitespace normalization;
- simple page artifact removal;
- broken hyphenation correction;
- empty or very short page removal.


In [6]:
from agro_rag.processing.clean_text import clean_parsed_pages

cleaned_pages = clean_parsed_pages(
    parsed_df=pages_df,
    text_column="text",
    output_column="clean_text",
    min_chars=30,
)

cleaned_pages_path = PARSED_DOCUMENTS_DIR / "all_pages_clean.parquet"
cleaned_pages.to_parquet(cleaned_pages_path, index=False)

print("Cleaned pages:", len(cleaned_pages))

cleaned_pages[
    ["source", "document_name", "page_number", "theme", "clean_text_length"]
].head()


Cleaned pages: 3


,source,document_name,page_number,theme,clean_text_length
0,MapBiomas,synthetic_mapbiomas_land_use_note,1,land use and land cover,288
1,INPE,synthetic_inpe_prodes_note,1,deforestation monitoring,296
2,Project limitations,synthetic_limitations_note,1,limitations,299


## 7. Create document chunks

The output is the core RAG document table:

```text
data/processed/document_chunks.parquet
```

Recommended initial chunking strategy:

```text
chunk_size = 250 words
chunk_overlap = 40 words
```


In [7]:
from agro_rag.processing.chunk_documents import chunk_pages_dataframe

chunks_df = chunk_pages_dataframe(
    pages_df=cleaned_pages,
    text_column="clean_text",
    chunk_size=250,
    chunk_overlap=40,
)

document_chunks_path = PROCESSED_DATA_DIR / "document_chunks.parquet"
chunks_df.to_parquet(document_chunks_path, index=False)

print("Chunks created:", len(chunks_df))

chunks_df[
    ["chunk_id", "source", "document_name", "page_number", "theme", "chunk_word_count"]
].head()


Chunks created: 3


,chunk_id,source,document_name,page_number,theme,chunk_word_count
0,mapbiomas__synthetic_mapbiomas_land_use_note__...,MapBiomas,synthetic_mapbiomas_land_use_note,1,land use and land cover,38
1,inpe__synthetic_inpe_prodes_note__p0001__c001,INPE,synthetic_inpe_prodes_note,1,deforestation monitoring,38
2,project_limitations__synthetic_limitations_not...,Project limitations,synthetic_limitations_note,1,limitations,41


## 8. Build the ChromaDB vector store

This step embeds the chunks and stores them in ChromaDB.

The vector database is generated locally and should not be committed to GitHub.


In [8]:
from agro_rag.rag.vector_store import build_vector_store_from_chunks

build_vector_store_from_chunks(
    chunks_path=document_chunks_path,
    persist_directory=CHROMA_DB_DIR,
    collection_name="agro_environmental_documents",
    reset_collection=True,
)

print("Vector store built successfully.")
print("ChromaDB directory:", CHROMA_DB_DIR)


C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aferr\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate

Vector store built successfully.
ChromaDB directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\chroma_db


## 9. Test vector retrieval

We now ask a question and inspect the retrieved chunks.


In [9]:
from agro_rag.rag.vector_store import query_vector_store, format_retrieved_context

question = "How can MapBiomas help identify native vegetation loss?"

retrieved_df = query_vector_store(
    question=question,
    persist_directory=CHROMA_DB_DIR,
    collection_name="agro_environmental_documents",
    n_results=3,
)

retrieved_df[
    ["rank", "source", "document_name", "theme", "distance", "chunk_text"]
]


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6623.23it/s]


,rank,source,document_name,theme,distance,chunk_text
0,1,MapBiomas,synthetic_mapbiomas_land_use_note,land use and land cover,0.454421,MapBiomas provides annual land use and land co...
1,2,INPE,synthetic_inpe_prodes_note,deforestation monitoring,1.226668,INPE and TerraBrasilis provide official defore...
2,3,Project limitations,synthetic_limitations_note,limitations,1.459552,This RAG system is designed for evidence organ...


In [10]:
context = format_retrieved_context(retrieved_df)
print(context)


[Retrieved chunk]
Source: MapBiomas
Document: synthetic_mapbiomas_land_use_note
Page: 1
Year: 2024
Theme: land use and land cover
Text: MapBiomas provides annual land use and land cover information. It can support analysis of forest area, native vegetation, agriculture, pasture, and land-use transitions. These indicators support environmental screening, but they do not prove illegal deforestation or legal non-compliance.

---

[Retrieved chunk]
Source: INPE
Document: synthetic_inpe_prodes_note
Page: 1
Year: 2024
Theme: deforestation monitoring
Text: INPE and TerraBrasilis provide official deforestation monitoring evidence. PRODES is commonly used for annual deforestation monitoring. DETER is alert-oriented and should not be interpreted in the same way as PRODES. Deforestation evidence should be interpreted with methodological documentation.

---

[Retrieved chunk]
Source: Project limitations
Document: synthetic_limitations_note
Page: 1
Year: 2024
Theme: limitations
Text: This RAG syste

## 10. Retrieval-only RAG test

This test does not call an LLM.

It verifies that the retrieval pipeline works.


In [11]:
from agro_rag.rag.qa_chain import answer_question_without_llm

retrieval_result = answer_question_without_llm(
    question="Can this system prove illegal deforestation?",
    persist_directory=CHROMA_DB_DIR,
    collection_name="agro_environmental_documents",
    n_results=3,
)

print(retrieval_result.answer)
print()
print(retrieval_result.retrieved_context)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4992.76it/s]


LLM generation was not executed. Retrieval-only mode was used.

[Retrieved chunk]
Source: INPE
Document: synthetic_inpe_prodes_note
Page: 1
Year: 2024
Theme: deforestation monitoring
Text: INPE and TerraBrasilis provide official deforestation monitoring evidence. PRODES is commonly used for annual deforestation monitoring. DETER is alert-oriented and should not be interpreted in the same way as PRODES. Deforestation evidence should be interpreted with methodological documentation.

---

[Retrieved chunk]
Source: MapBiomas
Document: synthetic_mapbiomas_land_use_note
Page: 1
Year: 2024
Theme: land use and land cover
Text: MapBiomas provides annual land use and land cover information. It can support analysis of forest area, native vegetation, agriculture, pasture, and land-use transitions. These indicators support environmental screening, but they do not prove illegal deforestation or legal non-compliance.

---

[Retrieved chunk]
Source: Project limitations
Document: synthetic_limitations

## 11. Optional LLM answer

This cell requires:

```text
OPENAI_API_KEY
```

If the key is not configured, the cell will skip the LLM call.


In [12]:
import os
from agro_rag.rag.qa_chain import answer_question

if os.getenv("OPENAI_API_KEY"):
    rag_result = answer_question(
        question="Can this system prove illegal deforestation?",
        persist_directory=CHROMA_DB_DIR,
        collection_name="agro_environmental_documents",
        n_results=3,
    )

    print(rag_result.answer)
else:
    print("OPENAI_API_KEY not found. Skipping LLM-based answer.")


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3062.14it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## 12. Generate golden questions

This creates:

```text
data/processed/rag_eval_questions.csv
```

These questions are used to evaluate retrieval and answer behavior.


In [13]:
from agro_rag.evaluation.golden_questions import save_golden_questions

golden_questions = save_golden_questions(
    output_path=PROCESSED_DATA_DIR / "rag_eval_questions.csv"
)

print("Golden questions:", len(golden_questions))
golden_questions.head()


Golden questions: 20


,question_id,question,expected_source,expected_theme,expected_metadata,expected_answer_type,unsafe_risk,notes
0,Q001,What does INPE / PRODES indicate about annual ...,INPE,deforestation,PRODES; annual monitoring,direct_answer,False,Should retrieve INPE or TerraBrasilis evidence...
1,Q002,What does MapBiomas indicate about land use an...,MapBiomas,land use and land cover,MapBiomas; land cover; land use,direct_answer,False,Should retrieve MapBiomas evidence about land-...
2,Q003,How can MapBiomas data help identify native ve...,MapBiomas,native vegetation,native vegetation; land-use transition,explanation,False,Should explain that MapBiomas supports land-co...
3,Q004,How can INPE and MapBiomas evidence complement...,MapBiomas; INPE,source comparison,PRODES; MapBiomas; land use; deforestation,comparison,False,Should compare official deforestation monitori...
4,Q005,What is the difference between deforestation e...,MapBiomas; INPE,methodology,deforestation; land-use transition,comparison,False,Should distinguish INPE deforestation monitori...


## 13. Run retrieval evaluation

This evaluates whether the retriever returns the expected source family.

For the synthetic demo, metrics are only illustrative. With real MapBiomas and INPE documents, the results become meaningful.


In [14]:
from agro_rag.evaluation.evaluate_retrieval import evaluate_retrieval_file

retrieval_eval = evaluate_retrieval_file(
    golden_questions_path=PROCESSED_DATA_DIR / "rag_eval_questions.csv",
    output_path=PROJECT_ROOT / "reports" / "rag_retrieval_evaluation.csv",
    persist_directory=CHROMA_DB_DIR,
    collection_name="agro_environmental_documents",
    n_results=3,
)

retrieval_eval.head()


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6598.54it/s]


Retrieval evaluation summary
----------------------------
Questions: 20
Source hit rate@3: 0.900
Mean precision@3: 0.450
Mean recall@3: 0.900


,question_id,question,expected_source,expected_theme,expected_answer_type,unsafe_risk,n_results,source_hit_at_k,recall_at_k,precision_at_k,retrieved_sources,retrieved_documents,top_chunk_id,top_distance
0,Q001,What does INPE / PRODES indicate about annual ...,INPE,deforestation,direct_answer,False,3,1,1,0.333333,INPE; MapBiomas; Project limitations,synthetic_inpe_prodes_note; synthetic_mapbioma...,inpe__synthetic_inpe_prodes_note__p0001__c001,0.484437
1,Q002,What does MapBiomas indicate about land use an...,MapBiomas,land use and land cover,direct_answer,False,3,1,1,0.333333,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,mapbiomas__synthetic_mapbiomas_land_use_note__...,0.574277
2,Q003,How can MapBiomas data help identify native ve...,MapBiomas,native vegetation,explanation,False,3,1,1,0.333333,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,mapbiomas__synthetic_mapbiomas_land_use_note__...,0.459135
3,Q004,How can INPE and MapBiomas evidence complement...,MapBiomas; INPE,source comparison,comparison,False,3,1,1,0.666667,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,mapbiomas__synthetic_mapbiomas_land_use_note__...,0.691901
4,Q005,What is the difference between deforestation e...,MapBiomas; INPE,methodology,comparison,False,3,1,1,0.666667,INPE; MapBiomas; Project limitations,synthetic_inpe_prodes_note; synthetic_mapbioma...,inpe__synthetic_inpe_prodes_note__p0001__c001,0.976538


## 14. Final checklist

After this notebook runs successfully, the project has a working document RAG foundation.

Generated files:

```text
data/interim/parsed_documents/all_pages_clean.parquet
data/processed/document_chunks.parquet
data/processed/rag_eval_questions.csv
reports/rag_retrieval_evaluation.csv
chroma_db/
```

Next practical steps:

1. add real MapBiomas and INPE PDFs;
2. rerun this notebook;
3. inspect retrieval quality;
4. launch the Streamlit app:

```bash
streamlit run app/streamlit_app.py
```
